# Simulating iDPC-STEM images with abTEM

This notebook is designed for performing iDPC-STEM simulations with abTEM.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt 

import ase as ase
from ase.io import read
from ase import Atoms
from ase.visualize import view
from ase.spacegroup import crystal

import abtem as ab
from abtem import *

import dask as dask

In [ ]:
from importlib.metadata import version
print(f"abTEM installation tested is {version('abtem')}, conda installation")

In [ ]:
import cupy as cp

print(cp.cuda.runtime.getDeviceCount())
print(cp.cuda.runtime.getDeviceProperties(0)["name"])

In [ ]:
#GPU/CPU parameters
ab.config.set({"device": "gpu"});
dask.config.set({"num_workers": 1});
ab.config.set({"dask.chunk-size-gpu": "2048 MB"})
ab.config.set({"cupy.fft-cache-size": "1024 MB"})
# set config according to your hardware capabilities.
# GPU will be quicker if you have CUDA capabilities.

## Function definitions:

In [ ]:
def intgrad2d(gradient: np.ndarray, sampling: tuple[float, float] = None):
    """
    Perform Fourier-space integration of gradient. 
    Taken from the legacy version of abTEM.
    Please see https://github.com/abTEM/abTEM-legacy

    Parameters
    gradient : two np.ndarrays
        The x- and y-components of the gradient.
    sampling : two float
        Lateral sampling of the gradients. Default is 1.0.

    Returns
    np.ndarray
        Integrated center of mass measurement
    """
    gx, gy = gradient
    (nx, ny) = gx.shape
    ikx = np.fft.fftfreq(nx, d=sampling[0])
    iky = np.fft.fftfreq(ny, d=sampling[1])
    grid_ikx, grid_iky = np.meshgrid(ikx, iky, indexing='ij')
    k = grid_ikx ** 2 + grid_iky ** 2
    k[k == 0] = 1e-12
    That = (np.fft.fft2(gx) * grid_ikx + np.fft.fft2(gy) * grid_iky) / (2j * np.pi * k)
    T = np.real(np.fft.ifft2(That))
    T -= T.min()
    return T


In [ ]:
def iDPC_conversion(dpc_measurement):
    """
    Author : Ella Aimee Kitching

    Perform transformation from DPC to iDPC.
    Use if taking data outside of the abTEM ecosystem.

    Parameters
    dpc_measurement: 
        The simulated DPC measurement using abTEM's SegmentedDetector.
    Returns
    np.ndarray
        Integrated DPC measurement 
    """
    
    differential = dpc_measurement.differentials(
        direction_1 = [((3,)),((1,)),],
        direction_2 = [((0,)),((2,)),],
    return_complex=False)
    
    differential_signal_x = differential[0]
    differential_signal_y = differential[1] 

    grad =  differential_signal_x.array, differential_signal_y.array

    idpc = intgrad2d(grad,[-1,-1])
    
    return idpc
    

In [ ]:
def iDPC_abtem(dpc_measurement):
    """    
    Author : Ella Aimee Kitching

    Perform transformation from DPC to iDPC.
    Use if keeping data inside of the abTEM ecosystem.

    Parameters
    dpc_measurement: abTEM measurement object
        The simulated DPC measurement using abTEM's SegmentedDetector.
    
    Returns
    np.ndarray
        Integrated DPC measurement
    """
    differentialComplex = dpc_measurement.differentials(
        direction_1 = [((2, 1,)),((0, 3,)),],
        direction_2 = [((2, 3,)),((0, 1,)),],
    return_complex=True)
    integrateTest = differentialComplex.integrate_gradient()
    
    return integrateTest

# Code starts here

In [ ]:
# loop
atoms = read(r"trun_ceria_111_100_3nm_OTermVac.cfg")
semiangle_Cut = 15.3
energy = 200e3
dpc_detector = SegmentedDetector(inner=11, outer=40, nbins_radial = 1, nbins_azimuthal = 4)

for k in range(-60, 66, 5):
    cp.get_default_memory_pool().free_all_blocks()
    
    atoms_rot = atoms.copy()
    atoms_rot.rotate(k, 'x', rotate_cell=False)
    atoms_rot.center(axis=(0,1,2))

    frozen_phonons = ab.FrozenPhonons(atoms_rot, num_configs=20, sigmas=0.1, seed=155)
    potential_phonon = ab.Potential(frozen_phonons, slice_thickness=2, gpts = (512,512)) # 335.6 corresponds to 19.08, as long it's larger its fine!
    gridscan = GridScan(start=[0, 0], end= potential_phonon.extent, sampling = (0.1908)) # corresponds to 19.08 pm 

    S_phonon = SMatrix(
        interpolation=4,
        energy=energy,
        semiangle_cutoff=semiangle_Cut,
        potential=potential_phonon
    )

    dpc_measurement_phonon = S_phonon.scan(gridscan, dpc_detector)

    # compute
    dpc_measurement_phonon.compute(num_workers=8)
    integratedDPC = iDPC_abtem(dpc_measurement_phonon)
    
    # set save path depending on angle
    savePathRaw = fr"sims\raw\angle_{k}"
    savePathNoise = fr"sims\noise\angle_{k}"

    filtered_measurements = integratedDPC.gaussian_filter(0.3)
    noisy_measurements = filtered_measurements.poisson_noise(dose_per_area=5e3)

    # save
    integratedDPC.to_zarr(savePathRaw, compute=True, overwrite=False)
    noisy_measurements.to_zarr(savePathNoise, compute=True, overwrite=False)
    